# FMCG Promotional ROI Analysis

This notebook analyses a synthetic FMCG promotional performance dataset.

The purpose is to evaluate whether promotions generated profitable incremental growth, not only sales uplift.

Key commercial questions:

- Which promotions generated profitable uplift?
- Which campaigns increased volume but reduced profit?
- Which promo mechanics performed best?
- Which products, categories, regions, and accounts responded best?
- Which promotions should be repeated, adjusted, reviewed, or discontinued?

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load Dataset

In [ ]:
possible_paths = [
    Path("../data/fmcg_promo_roi_synthetic_dataset.csv"),
    Path("data/fmcg_promo_roi_synthetic_dataset.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Dataset not found. Please check the data folder path.")

df = pd.read_csv(data_path)
df["Week_Start"] = pd.to_datetime(df["Week_Start"])
df.head()

## 3. Dataset Overview

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Date range: {df['Week_Start'].min().date()} to {df['Week_Start'].max().date()}")
print(f"Promo rows: {df['Promo_Flag'].sum():,}")
print(f"Non-promo rows: {(df['Promo_Flag'] == 0).sum():,}")

df.info()

## 4. Data Quality Checks

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

In [ ]:
checks = {
    "negative_baseline_units": (df["Baseline_Units"] < 0).sum(),
    "negative_actual_units": (df["Actual_Units"] < 0).sum(),
    "promo_price_above_regular_price": (df["Promo_Price"] > df["Regular_Price"]).sum(),
    "unit_cost_above_selling_price": (df["Unit_Cost"] > df["Selling_Price"]).sum(),
}
checks

## 5. Validate Core Metric Calculations

In [ ]:
df["Check_Uplift_Units"] = df["Actual_Units"] - df["Baseline_Units"]
df["Check_Baseline_Gross_Profit"] = df["Baseline_Units"] * (df["Regular_Price"] - df["Unit_Cost"])
df["Check_Gross_Profit_Before_Promo_Cost"] = df["Actual_Units"] * (df["Selling_Price"] - df["Unit_Cost"])
df["Check_Incremental_Profit_After_Promo_Cost"] = (
    df["Check_Gross_Profit_Before_Promo_Cost"]
    - df["Check_Baseline_Gross_Profit"]
    - df["Promo_Cost"]
)

validation = pd.DataFrame({
    "metric": [
        "Uplift Units",
        "Baseline Gross Profit",
        "Gross Profit Before Promo Cost",
        "Incremental Profit After Promo Cost"
    ],
    "max_absolute_difference": [
        (df["Uplift_Units"] - df["Check_Uplift_Units"]).abs().max(),
        (df["Baseline_Gross_Profit"] - df["Check_Baseline_Gross_Profit"]).abs().max(),
        (df["Gross_Profit_Before_Promo_Cost"] - df["Check_Gross_Profit_Before_Promo_Cost"]).abs().max(),
        (df["Incremental_Profit_After_Promo_Cost"] - df["Check_Incremental_Profit_After_Promo_Cost"]).abs().max()
    ]
})
validation

## 6. Executive Summary

In [ ]:
promo = df[df["Promo_Flag"] == 1].copy()

executive_summary = pd.DataFrame({
    "Metric": [
        "Promotional events",
        "Total baseline units",
        "Total actual units",
        "Total uplift units",
        "Average uplift %",
        "Total actual revenue",
        "Total promo cost",
        "Total incremental profit after promo cost",
        "Median promo ROI",
        "Positive ROI rate",
        "High cannibalisation risk events"
    ],
    "Value": [
        len(promo),
        promo["Baseline_Units"].sum(),
        promo["Actual_Units"].sum(),
        promo["Uplift_Units"].sum(),
        promo["Uplift_Pct"].mean(),
        promo["Actual_Revenue"].sum(),
        promo["Promo_Cost"].sum(),
        promo["Incremental_Profit_After_Promo_Cost"].sum(),
        promo["Promo_ROI"].median(),
        (promo["Promo_ROI"] > 0).mean(),
        (promo["Cannibalisation_Risk"] == "High").sum()
    ]
})
executive_summary

## 7. Promo Performance by Type

In [ ]:
promo_type_summary = (
    promo.groupby("Promo_Type")
    .agg(
        promo_events=("Campaign_ID", "count"),
        avg_uplift_pct=("Uplift_Pct", "mean"),
        total_uplift_units=("Uplift_Units", "sum"),
        total_actual_revenue=("Actual_Revenue", "sum"),
        total_promo_cost=("Promo_Cost", "sum"),
        total_incremental_profit=("Incremental_Profit_After_Promo_Cost", "sum"),
        median_roi=("Promo_ROI", "median"),
        positive_roi_rate=("Promo_ROI", lambda x: (x > 0).mean())
    )
    .sort_values("total_incremental_profit", ascending=False)
)
promo_type_summary

## 8. Category Performance

In [ ]:
category_summary = (
    promo.groupby("Category")
    .agg(
        promo_events=("Campaign_ID", "count"),
        total_uplift_units=("Uplift_Units", "sum"),
        avg_uplift_pct=("Uplift_Pct", "mean"),
        total_incremental_revenue=("Incremental_Revenue_From_Uplift", "sum"),
        total_promo_cost=("Promo_Cost", "sum"),
        total_incremental_profit=("Incremental_Profit_After_Promo_Cost", "sum"),
        median_roi=("Promo_ROI", "median"),
        avg_rate_of_sale=("Rate_of_Sale_Units_per_Store_per_Week", "mean")
    )
    .sort_values("total_incremental_profit", ascending=False)
)
category_summary

## 9. Region and Account Performance

In [ ]:
region_account_summary = (
    promo.groupby(["Region", "Account"])
    .agg(
        promo_events=("Campaign_ID", "count"),
        total_uplift_units=("Uplift_Units", "sum"),
        avg_uplift_pct=("Uplift_Pct", "mean"),
        total_incremental_profit=("Incremental_Profit_After_Promo_Cost", "sum"),
        median_roi=("Promo_ROI", "median"),
        avg_rate_of_sale=("Rate_of_Sale_Units_per_Store_per_Week", "mean")
    )
    .sort_values("total_incremental_profit", ascending=False)
)
region_account_summary.head(15)

## 10. Identify High-Uplift but Low-Profit Promotions

In [ ]:
uplift_threshold = promo["Uplift_Pct"].quantile(0.75)

high_uplift_low_profit = (
    promo[
        (promo["Uplift_Pct"] >= uplift_threshold)
        & (promo["Incremental_Profit_After_Promo_Cost"] <= 0)
    ]
    .sort_values(["Uplift_Pct", "Incremental_Profit_After_Promo_Cost"], ascending=[False, True])
)

high_uplift_low_profit[
    [
        "Campaign_ID", "Product", "Category", "Region", "Account", "Promo_Type",
        "Discount_Pct", "Uplift_Pct", "Promo_ROI", "Promo_Cost",
        "Incremental_Profit_After_Promo_Cost", "Cannibalisation_Risk", "Recommendation"
    ]
].head(15)

## 11. Promotions to Repeat

In [ ]:
repeat_candidates = (
    promo[
        (promo["Recommendation"].isin(["Repeat", "Targeted repeat"]))
        & (promo["Promo_ROI"] > 0)
        & (promo["Incremental_Profit_After_Promo_Cost"] > 0)
    ]
    .sort_values("Incremental_Profit_After_Promo_Cost", ascending=False)
)

repeat_candidates[
    [
        "Campaign_ID", "Product", "Category", "Region", "Account", "Promo_Type",
        "Uplift_Pct", "Promo_ROI", "Incremental_Profit_After_Promo_Cost",
        "Cannibalisation_Risk", "Recommendation"
    ]
].head(15)

## 12. Promotions to Adjust or Discontinue

In [ ]:
review_candidates = (
    promo[promo["Recommendation"].isin(["Adjust depth/mechanic", "Discontinue", "Review"])]
    .sort_values("Incremental_Profit_After_Promo_Cost")
)

review_candidates[
    [
        "Campaign_ID", "Product", "Category", "Region", "Account", "Promo_Type",
        "Discount_Pct", "Uplift_Pct", "Margin_Impact_pp", "Promo_ROI",
        "Incremental_Profit_After_Promo_Cost", "Cannibalisation_Risk", "Recommendation"
    ]
].head(15)

## 13. Visualisation: Median Promo ROI by Promo Type

In [ ]:
roi_by_type = promo.groupby("Promo_Type")["Promo_ROI"].median().sort_values()

plt.figure(figsize=(10, 5))
roi_by_type.plot(kind="bar")
plt.title("Median Promo ROI by Promo Type")
plt.xlabel("Promo Type")
plt.ylabel("Median Promo ROI")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 14. Visualisation: Uplift % vs Promo ROI

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    promo["Uplift_Pct"],
    promo["Promo_ROI"],
    s=np.clip(promo["Incremental_Revenue_From_Uplift"] / 100, 10, 250),
    alpha=0.5
)
plt.axhline(0, linestyle="--")
plt.title("Uplift % vs Promo ROI")
plt.xlabel("Uplift %")
plt.ylabel("Promo ROI")
plt.tight_layout()
plt.show()

## 15. Visualisation: Incremental Profit by Category

In [ ]:
profit_by_category = promo.groupby("Category")["Incremental_Profit_After_Promo_Cost"].sum().sort_values()

plt.figure(figsize=(9, 5))
profit_by_category.plot(kind="bar")
plt.title("Total Incremental Profit by Category")
plt.xlabel("Category")
plt.ylabel("Incremental Profit After Promo Cost")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 16. Recommendation Summary

In [ ]:
recommendation_summary = (
    promo.groupby("Recommendation")
    .agg(
        promo_events=("Campaign_ID", "count"),
        avg_uplift_pct=("Uplift_Pct", "mean"),
        total_uplift_units=("Uplift_Units", "sum"),
        total_promo_cost=("Promo_Cost", "sum"),
        total_incremental_profit=("Incremental_Profit_After_Promo_Cost", "sum"),
        median_roi=("Promo_ROI", "median"),
        high_cannibalisation_events=("Cannibalisation_Risk", lambda x: (x == "High").sum())
    )
    .sort_values("total_incremental_profit", ascending=False)
)
recommendation_summary

## 17. Commercial Interpretation Template

In [ ]:
commercial_insights = [
    "Promotions should not be judged by uplift alone; several high-uplift promotions may generate weak or negative ROI after margin impact and promo cost.",
    "Repeat candidates should have positive incremental profit, positive ROI, and manageable cannibalisation risk.",
    "Targeted repeat campaigns may be appropriate where performance is concentrated in specific regions, accounts, or categories.",
    "Promotions with high uplift but negative incremental profit should be adjusted through lower discount depth, different mechanics, or improved retailer funding.",
    "High cannibalisation risk campaigns should be reviewed with category context before being repeated."
]
for insight in commercial_insights:
    print(f"- {insight}")

## 18. Next Steps

Suggested next steps:

1. Export summary tables for Power BI or Excel.
2. Build dashboard pages: Executive Summary, Promo ROI by Campaign, Uplift vs Margin Impact, Category / Region Analysis, and Cannibalisation Risk.
3. Add dashboard screenshots to the README.
4. Finalise recommendations for repeat, targeted repeat, adjust, review, and discontinue.

In [ ]:
output_dir = Path("../data") if Path("../data").exists() else Path(".")
promo_type_summary.to_csv(output_dir / "promo_type_summary.csv")
category_summary.to_csv(output_dir / "category_summary.csv")
region_account_summary.to_csv(output_dir / "region_account_summary.csv")
recommendation_summary.to_csv(output_dir / "recommendation_summary.csv")
print("Exported summary tables.")